# Konu 12 — Araç kullanımı: modelin karar verdiği program

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Şimdiye kadar hangi fonksiyonun ne zaman çalışacağına sen karar verdin: `ciz`'i sen çağırdın, sırayı sen yazdın.
Bugün bir kararı **modele** bırakıyoruz. Kullanıcı serbest bir cümle yazacak ("şu fotoğrafın arka planını sadeleştir");
model iki araçtan hangisinin gerektiğine karar verecek, programımız o aracı çalıştıracak. Sonra modelin kararlarını
**ölçeceğiz**: kaçında seninle aynı fikirde?

Isınma anahtarsız. Adım 1'den sonrası `anahtar.txt` ister (Konu 2'deki dosya). Defterin tamamı metin modeline yaklaşık
33 istek gönderir ve en fazla 5 görsel üretir ya da düzenler; günlük ücretsiz kotanın %10'undan az. Ölçüm
adımları (Adım 6–7) **görsel üretmez**, yalnızca karar sorar.

## Isınma — Kararı önce sen ver

Elimizde iki araç olacak: **uret** sıfırdan yeni bir görsel üretir, **duzenle** elimizdeki bir görseli değiştirir.
Elimizdeki tek görsel bir fincan fotoğrafı (Adım 1'de göreceksin).

Aşağıda kullanıcılardan gelmiş altı istek var. Model karar vermeden önce **sen** karar ver: her istek için `uret` mi,
`duzenle` mi? Kararlarını `etiketlerim` listesine aynı sırayla yaz. Listede hocanın etiketleri duruyor; katılmadığın
yeri değiştir.

In [ ]:
istekler = [
    "kafe için yeni bir afiş görseli yap",
    "şu fincan fotoğrafının arka planını sadeleştir",
    "menü kapağı için bir görsel üret",
    "bu görseldeki renkleri daha sıcak yap",
    "afişi daha sıcak yap",
    "yaz kampanyası için bir görsel",
]
etiketlerim = ["uret", "duzenle", "uret", "duzenle", "uret", "uret"]

print(len(istekler), len(etiketlerim))

İki sayı da 6 olmalı: her isteğin bir etiketi var. Şimdi yan yana yazdıralım. İki listeyi aynı anda gezmek için
bir **sıra sayacı** tutuyoruz: `sira` 0'dan başlar, her turda bir artar; `etiketlerim[sira]` o sıradaki etiket.

In [ ]:
sira = 0
for istek in istekler:
    print(etiketlerim[sira], "<-", istek)
    sira = sira + 1

**Sohbete yaz:** hangi istekte karar vermek zor geldi? Neden?

Bu etiketler Konu 3'teki **test verisi** gibi: doğru cevabı insan verdi. Adım 6'da modelin kararlarını bunlarla
karşılaştıracağız.

## Adım 1 — Araçlar ve elimizdeki görsel

`gorsel_araclari.py` bu klasörde duran yardımcı dosya. İçindeki iki fonksiyon modelin seçeceği **araçlar**:

- `ciz(yonerge, tohum)`: Konu 8'de yazdığın, Konu 9'da kullandığın fonksiyon. Yeni bir görsel üretir.
- `duzenle(dosya, yonerge, tohum)`: Konu 8 Adım 9'daki düzenleme isteğinin fonksiyon hâli. Görseli en fazla 512 piksele
  küçültür, form olarak gönderir.

İkisi de görselin baytlarını verir; başarısızsa `None`. Model bu fonksiyonları değil, araçların **adlarını** görecek:
yeni görsel için `uret`, düzenleme için `duzenle`. Hangi adın hangi fonksiyonu çalıştıracağını Adım 5'te biz yazacağız. `adres` ve `basliklar` metin modeline istek gönderirken lazım.
Anahtarı `../anahtar.txt`'den yardımcı dosya kendisi okur.

In [ ]:
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from gorsel_araclari import ciz, duzenle, adres, basliklar

Metin modeli Konu 2'den beri kullandığımız model:

In [ ]:
MODEL = "@cf/meta/llama-3.3-70b-instruct-fp8-fast"
metin_adresi = adres(MODEL)

Elimizdeki görsel: `veri/fincan.jpg` (Konu 8'deki fincan). `duzenle` hep bu görseli değiştirecek.

In [ ]:
fincan = plt.imread("veri/fincan.jpg")
plt.imshow(fincan)
plt.axis("off")
plt.show()

## Adım 2 — Alanları belli cevap: JSON modu

Konu 2'de modelden bir cevap istedik, bir paragraf geldi. Paragrafı insan okur; program okuyamaz. Program için
cevabın **alanları belli** olmalı: bir sözlük gibi. Buna **yapılandırılmış çıktı** denir.

Cloudflare'de bunun yolu gövdeye bir alan daha eklemek: `"response_format"` (cevabın biçimi).
`{"type": "json_object"}` "cevabı JSON olarak ver" demek. Hangi alanları istediğimizi sorunun içinde yazıyoruz.

In [ ]:
bicim = {"type": "json_object"}
soru = "Sessiz bir çalışma kafesi için bir slogan öner. Cevabı JSON ver; alanlar: slogan, ton, hedef_kitle."
govde = {"prompt": soru, "response_format": bicim}

cevap = requests.post(metin_adresi, headers=basliklar, json=govde, timeout=60)
print(cevap.status_code)

`200` yazmadıysa aşağıdaki hücreyi çalıştırma; durum koduna bak (Konu 2). Model istenen biçime uyamazsa da hata
dönebilir: JSON modu biçimi garanti etmez.

`200` ise cevabı Konu 2'deki gibi iki katmanda açıyoruz. Fark `sonuc["response"]`'ta: bu kez bir paragraf değil,
bir **sözlük**.

In [ ]:
yanit = cevap.json()
sonuc = yanit["result"]
kart = sonuc["response"]
print(kart)
print(kart["ton"])

Cevabın bir alanına adıyla erişebiliyoruz: `kart["ton"]`. Konu 9'daki kartlar gibi.

Bu isteği çok kez göndereceğiz; bir fonksiyona koyalım. Konu 5'teki `modele_sor` gibi sıcaklığı (`sicaklik`) da alıyor.
İstek başarısızsa `None` veriyor.

In [ ]:
def json_sor(soru, sicaklik):
    govde = {"prompt": soru, "temperature": sicaklik, "response_format": bicim}
    cevap = requests.post(metin_adresi, headers=basliklar, json=govde, timeout=60)
    if cevap.status_code != 200:
        print("İstek başarısız, durum kodu:", cevap.status_code)
        return None
    yanit = cevap.json()
    sonuc = yanit["result"]
    return sonuc["response"]

## Adım 3 — Metin artık veri

Aynı soruyu 5 kez soruyoruz (`range(5)`: döngü 5 tur döner) ve kartları bir listede topluyoruz. Sıcaklık 0.9: her seferinde
farklı bir slogan gelsin. Birkaç saniye sürer.

In [ ]:
kartlar = []
for tur in range(5):
    kart = json_sor(soru, 0.9)
    kartlar.append(kart)
    print(kart["ton"], "|", kart["slogan"])

Kartlar artık **veri**: bir tabloya dökülebilir, süzülebilir. Konu 1'deki gibi süzelim: yalnızca tonu `samimi` olanlar.

**Çalıştırmadan önce tahmin et:** kaç slogan yazılacak? Üstteki `ton` değerlerine bak.

In [ ]:
for kart in kartlar:
    if kart["ton"] == "samimi":
        print(kart["slogan"])

`ton` serbest bir alan: model her seferinde başka bir kelime yazabilir ("samimi", "Samimi", "sıcak ve samimi").
Program için bunların hepsi farklı. Bir form tasarlarken serbest metin kutusu yerine **açılır menü** koymak gibi,
alanın alabileceği değerleri soruda sınırlıyoruz:

In [ ]:
soru = "Sessiz bir çalışma kafesi için bir slogan öner. Cevabı JSON ver; alanlar: slogan, ton (yalnızca samimi, resmi ya da esprili), hedef_kitle."

kartlar = []
for tur in range(5):
    kart = json_sor(soru, 0.9)
    kartlar.append(kart)
    print(kart["ton"], "|", kart["slogan"])

print("Samimi olanlar:")
for kart in kartlar:
    if kart["ton"] == "samimi":
        print(kart["slogan"])

Değerler artık üç seçenekten biri mi? Model kurala uymazsa süzgeç o kartı yine kaçırır; o yüzden çıktıya her zaman bak.

Bir sonraki adımda tam olarak bunu yapacağız: alanın değerini sınırlı bir listeden seçtireceğiz. Liste bu kez
tonlar değil, **araçlar**.

## Adım 4 — Karar da bir alan

Modele araçlarımızı **tarif ediyoruz**: adları ve ne yaptıkları. Bu tarif bilerek kısa ve biraz muğlak; Adım 7'de onu
düzelteceğiz.

In [ ]:
araclar = "uret: görsel üretir. duzenle: görseli düzenler."

Soruyu parça parça kuruyoruz (her satır soruya bir cümle ekler): önce araçlar, sonra ne istediğimiz, sonra cevabın
alanları, en sonda kullanıcının isteği. İki alan istiyoruz:

- `arac`: aracın adı (`uret` ya da `duzenle`). Bu, modelin **kararı**.
- `yonerge`: seçilen araca gidecek İngilizce yönerge. Modelin kullanıcının Türkçe cümlesinden çıkardığı.

`tarif` ayrı bir bilgi olarak geliyor; Adım 7'de başka bir tarifle aynı fonksiyonu çağıracağız. Sıcaklık 0: aynı
isteğe her seferinde aynı karar gelsin (Konu 5).

In [ ]:
def karar_ver(istek, tarif):
    soru = "Elindeki araçlar: " + tarif
    soru = soru + " Kullanıcının isteğine en uygun aracı seç."
    soru = soru + " Cevabı JSON ver; alanlar: arac (yalnızca aracın adı), yonerge (görsel modeline gidecek kısa İngilizce yönerge)."
    soru = soru + " Kullanıcının isteği: " + istek
    return json_sor(soru, 0)

In [ ]:
karar = karar_ver("şu fincan fotoğrafının arka planını sadeleştir", araclar)
print(karar)
print("Model şunu seçti:", karar["arac"])
print("Yönerge:", karar["yonerge"])

Kararı ekrana yazdırmak bu konunun alışkanlığı: modelin ne seçtiğini **görmeden** ne yaptığını bilemezsin.
Model hangi aracı seçti? Yönergeyi nasıl yazdı: kullanıcının cümlesinin çevirisi mi, daha fazlası mı?

## Adım 5 — Program kararı uygular

Model hiçbir şey çizmedi; yalnızca bir sözlük gönderdi. Kararı uygulayan **bizim programımız**: `arac` alanına bakıp
doğru fonksiyonu çağırıyor. `if / elif / else` GİTA2112'den tanıdık.

**Çalıştırmadan önce tahmin et:** model `"uret"` yerine `"üret"` (ü ile) yazarsa ne olur?

Önce çıktı klasörü:

In [ ]:
klasor = Path("ciktilar")
klasor.mkdir(exist_ok=True)

In [ ]:
def uygula(karar, dosya_adi):
    arac = karar["arac"]
    yonerge = karar["yonerge"]
    print("Model şunu seçti:", arac)
    print("Yönerge:", yonerge)
    if arac == "uret":
        resim = ciz(yonerge, 42)
    elif arac == "duzenle":
        resim = duzenle("veri/fincan.jpg", yonerge, 42)
    else:
        print("Bilinmeyen araç:", arac)
        resim = None
    if resim is None:
        print("Görsel yok")
        return None
    yol = klasor / dosya_adi
    with open(yol, "wb") as dosya:
        dosya.write(resim)
    print("Kaydedildi:", yol)
    return yol

`else` dalı bir **güvenlik ağı**: model listede olmayan bir ad yazarsa (`"üret"`, `"edit"`) program yanlış aracı
çalıştırmaz, durumu söyler. Modelin cevabına körü körüne güvenme.

Adım 4'teki kararı uyguluyoruz (birkaç saniye sürer):

In [ ]:
yol = uygula(karar, "karar_1.jpg")

`Kaydedildi` yazdıysa görsele bakalım (`Görsel yok` yazdıysa bu hücre hata verir; üstteki mesaja bak):

In [ ]:
gorsel = plt.imread(yol)
plt.imshow(gorsel)
plt.axis("off")
plt.show()

Şimdi başka türden bir istek. Aynı iki fonksiyon, aynı iki satır; bu kez model hangi aracı seçecek?

In [ ]:
karar = karar_ver("kafe için yeni bir afiş görseli yap", araclar)
yol = uygula(karar, "karar_2.jpg")

In [ ]:
gorsel = plt.imread(yol)
plt.imshow(gorsel)
plt.axis("off")
plt.show()

Bugün "yapay zeka ajanı" denen şeyin çekirdeği bu: model **karar verir**, program **uygular**. Gizemli bir şey yok:
bir sözlük ve bir `if`.

## Adım 6 — Model ne kadar doğru seçiyor?

Model iki örnekte doğru seçmiş olabilir, ama iki örnek bir şey kanıtlamaz. Isınmadaki altı isteğin hepsini soralım ve
kararları senin etiketlerinle karşılaştıralım: Konu 3'teki **doğruluk** (doğru / toplam).

Bu adımda görsel üretmiyoruz; yalnızca karar soruyoruz. Her istek tek bir metin isteği.

`olc` fonksiyonu bir tarif, bir istek listesi ve bir etiket listesi alıyor (Adım 7'de başka listelerle çağıracağız).
Isınmadaki sıra sayacı burada da var: `etiketler[sira]` o isteğin senin etiketin.

In [ ]:
def olc(tarif, liste, etiketler):
    sira = 0
    dogru = 0
    for istek in liste:
        karar = karar_ver(istek, tarif)
        arac = karar["arac"]
        beklenen = etiketler[sira]
        if arac == beklenen:
            dogru = dogru + 1
        else:
            print("Ayrıştık:", istek, "| ben:", beklenen, "| model:", arac)
        sira = sira + 1
    print("Doğru:", dogru, "/", len(liste))

**Çalıştırmadan önce tahmin et, sohbete yaz:** model hangi isteklerde senden ayrışacak? Kaç doğru bekliyorsun?

In [ ]:
olc(araclar, istekler, etiketlerim)

Ayrıştığınız istekler, Isınma'da senin de zorlandığın istekler mi? Model mi yanıldı, yoksa istek mi iki türlü
okunabiliyor? Modelin elinde yalnızca `araclar` metni var: "görseli düzenler" hangi görseli?

## Adım 7 — Sorun çoğu zaman tarifte

Bir tasarımcıya "afişi daha sıcak yap" desen o da sorar: hangi afiş? Modele verdiğimiz tarif de bir **brif**.
Muğlak brif, ister insana ister modele verilsin, yanlış işi getirir. Tarifi netleştiriyoruz: `duzenle` ne zaman
seçilmeli, emin değilse ne yapmalı.

In [ ]:
yeni_araclar = "uret: sıfırdan yeni bir görsel üretir."
yeni_araclar = yeni_araclar + " duzenle: YALNIZCA elimizdeki tek görseli, bir fincan fotoğrafını değiştirir."
yeni_araclar = yeni_araclar + " Kullanıcı 'bu', 'şu' ya da 'fotoğraf' diyerek o görseli işaret ediyorsa duzenle seç."
yeni_araclar = yeni_araclar + " Emin değilsen uret seç."
print(yeni_araclar)

Aynı altı istekle yeniden ölçüyoruz:

In [ ]:
olc(yeni_araclar, istekler, etiketlerim)

Sonuç değişti mi? Hangi istekte?

Dikkat: tarifi **bu altı isteğe bakarak** düzelttik ve yine bu altı istekle ölçtük. Konu 3'ü hatırla: test sonucuna
bakıp ayar yapmak, sonra aynı testte ölçmek, ölçümü güvenilmez yapar. Tarif yeni isteklerde de işe yarıyor mu?
Tarifi yazarken bakmadığımız beş istek; önce yine sen etiketle (listede hocanın etiketleri var; katılmadığın yeri değiştir).

In [ ]:
yeni_istekler = [
    "bu fincanın fotoğrafını siyah beyaz yap",
    "kahve festivali için bir poster tasarla",
    "fotoğraftaki fincanı kırmızı yap",
    "yeni şubemiz için bir tanıtım görseli hazırla",
    "görseli biraz daha aydınlat",
]
yeni_etiketlerim = ["duzenle", "uret", "duzenle", "uret", "duzenle"]

olc(yeni_araclar, yeni_istekler, yeni_etiketlerim)

Son isteğe bak: "görseli" diyor ama "bu", "şu" ya da "fotoğraf" demiyor. Kuralımıza göre model hangi aracı seçmeli? Kuralı
fazla dar yazmak da bir hata. Tarif yazmak bir kez yapılıp biten iş değil: yaz, ölç, ayrışan isteğe bak, düzelt, **yeni**
isteklerle yeniden ölç.

**Sohbete yaz:** kendi projen için tek cümlelik bir araç tarifi yaz. Hangi isteği yanlış araca gönderebilir?

## Adım 8 — Serbest cümleyle çalışan araç

Şimdi parçaları birleştiriyoruz: kullanıcı bir cümle yazar, model karar verir, program uygular. Bunu bir döngüde
yapıyoruz; her döngünün bir **durma koşulu** olmalı. Burada iki tane var:

- `range(3)`: en fazla 3 tur (3 kez). Konu 9'daki `MAKS_URETIM` gibi bir sınır.
- Boş bırakıp Enter'a basarsan `break` döngüyü bitirir.

`input` kutusuna kendi isteğini Türkçe yaz. Her tur bir karar ve bir görsel demek. Dosya adına tur numarası
yazılıyor (`istek_0.jpg`, `istek_1.jpg`...; sayma 0'dan başlıyor).

In [ ]:
for tur in range(3):
    istek = input("Ne istiyorsun? (bitirmek için boş bırak) ")
    if istek == "":
        print("Boş istek, duruyorum")
        break
    karar = karar_ver(istek, yeni_araclar)
    uygula(karar, f"istek_{tur}.jpg")

`ciktilar/` klasörüne bak. Model senin cümlelerinde doğru aracı seçti mi?

Bu döngüde her turdan sonra karar yine **senin** elinde: yeni bir cümle yazıyorsun ya da duruyorsun. Gerçek
"ajanlarda" aracın sonucu modele geri gider ve model bir sonraki adıma kendisi karar verir; sonucu beğenmezse
yeniden düzenler, yine beğenmez... Durma koşulu olmayan böyle bir döngü kotayı bitirene kadar dönebilir.
O yüzden ilk yazılan satır hep sınırdır.

## Bonus — Gerçek `tools` alanı (hoca gösterimi)

Adım 4'te kararı bir JSON alanı olarak **kendimiz** istedik. Modeller bunun standart bir yolunu da sunar: araçlar
gövdedeki `tools` alanında tanımlanır, model cevabında `tool_calls` (araç çağrıları) listesi döner. Fikir aynı:
model aracı seçer, çalıştıran yine biziz.

Her aracın bir adı (`name`), bir tarifi (`description`) ve aldığı bilgiler (`parameters`) var. Bilgilerin biçimi
iç içe sözlükler; her katmanı ayrı satırda kuruyoruz. İki aracımız da aynı bilgiyi alıyor: `yonerge`.
`"string"` bilginin metin olduğunu, `"required"` hangi bilgilerin zorunlu olduğunu söyler.

In [ ]:
yonerge_alani = {"type": "string", "description": "görsel modeline gidecek kısa İngilizce yönerge"}
bilgiler = {"yonerge": yonerge_alani}
parametreler = {"type": "object", "properties": bilgiler, "required": ["yonerge"]}

uret_araci = {"name": "uret", "description": "Sıfırdan yeni bir görsel üretir.", "parameters": parametreler}
duzenle_tarifi = "Elimizdeki fincan fotoğrafını değiştirir; yalnızca kullanıcı o görseli işaret ediyorsa."
duzenle_araci = {"name": "duzenle", "description": duzenle_tarifi, "parameters": parametreler}

Bu kez soru `prompt` ile değil, sohbet biçiminde (`messages`) gidiyor: bir mesaj listesi, her mesajın bir rolü (`user`:
kullanıcı) ve içeriği var. Model araç çağırmaya karar vermezse `tool_calls` gelmez; o zaman cevabı metin olarak yazdırıyoruz.

In [ ]:
mesaj = {"role": "user", "content": "şu fincan fotoğrafının arka planını sadeleştir"}
govde = {"messages": [mesaj], "tools": [uret_araci, duzenle_araci]}
cevap = requests.post(metin_adresi, headers=basliklar, json=govde, timeout=60)
print(cevap.status_code)

yanit = cevap.json()
sonuc = yanit["result"]
if "tool_calls" in sonuc:
    cagrilar = sonuc["tool_calls"]
    ilk_cagri = cagrilar[0]
    print("Model şunu çağırmak istiyor:", ilk_cagri["name"])
    print("Bilgiler:", ilk_cagri["arguments"])
else:
    print("Araç çağrılmadı; model şunu yazdı:", sonuc["response"])

Gerçek ajanlarda bundan sonra aracın sonucu ("görseli kaydettim") yeni bir mesaj olarak modele geri gönderilir ve
model bir sonraki adıma karar verir: **çağrı döngüsü**. Bu döngüye de Adım 8'deki gibi bir sınır konur
(ör. `for tur in range(3)`: en fazla 3 tur).